# Train A PEFT Mimi Speech-To-Speech Model

This notebook clones the full `kyutai-labs/moshi` repo, installs the repo dependencies, writes a JSON training config, and runs `scripts/train_peft_mimi_s2s.py`.

Prerequisites:
- one or more Mimi-tokenized Hugging Face dataset repos already built by `build_mimi_hf_dataset.py`
- access to the base model you choose
- a GPU runtime is strongly recommended

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kyutai-labs/moshi.git"
WORKSPACE_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORKSPACE_ROOT / "moshi"

if REPO_DIR.exists():
    print(f"Repo already exists at {REPO_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "status", "-sb"], check=True)
print(f"Working from: {REPO_DIR}")

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "moshi" / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR / "moshi")], check=True)

try:
    import torch
    print("torch", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch check failed:", exc)

In [ ]:
import getpass
import os

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")

print("HF token configured:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
import json

TOKEN_DATA_REPOS = [
    "yigagilbert/salt-s2s-lug-eng-studio-dataset_mimi_token_version",
]

TRAIN_OUTPUT_DIR = REPO_DIR / "artifacts" / "training" / "lora_mimi_s2s_llama"
SNAPSHOT_DIR = REPO_DIR / "artifacts" / "hf_mimi_snapshots"
CONFIG_PATH = REPO_DIR / "configs" / "mimi_s2s_train.notebook.json"

BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
TASK_TOKEN = "<TASK_S2ST_EN_LG>"
MAX_SEQ_LEN = 4096
TRUNCATE_POLICY = "tail"
CONTEXT_OVERFLOW = "clamp"
MAX_STEPS = 500
LR = 1e-4
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRAD_ACCUM = 8
EVAL_EVERY = 200
SAVE_EVERY = 500
NUM_WORKERS = 2
NUM_CODEBOOKS = 8
CARDINALITY = 2048
GRADIENT_CHECKPOINTING = True

config = {
    "hf_token_env": "HF_TOKEN",
    "data": {
        "snapshot_dir": str(SNAPSHOT_DIR),
        "repos": TOKEN_DATA_REPOS,
    },
    "training": {
        "output_dir": str(TRAIN_OUTPUT_DIR),
        "base_model": BASE_MODEL,
        "task_token": TASK_TOKEN,
        "max_seq_len": MAX_SEQ_LEN,
        "context_overflow": CONTEXT_OVERFLOW,
        "truncate_policy": TRUNCATE_POLICY,
        "max_steps": MAX_STEPS,
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "eval_every": EVAL_EVERY,
        "save_every": SAVE_EVERY,
        "num_workers": NUM_WORKERS,
        "num_codebooks": NUM_CODEBOOKS,
        "cardinality": CARDINALITY,
    },
}

CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(CONFIG_PATH)
print(json.dumps(config, indent=2))

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
for repo_id in TOKEN_DATA_REPOS:
    repo_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
    print(f"{repo_id}: {len(repo_files)} files")
    for required_name in ["dataset.train.jsonl", "dataset.validation.jsonl", "stats.json"]:
        print("  ", required_name, required_name in repo_files)

In [ ]:
import os
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    str(REPO_DIR / "scripts" / "train_peft_mimi_s2s.py"),
    "--config", str(CONFIG_PATH),
]

if GRADIENT_CHECKPOINTING:
    cmd.append("--gradient-checkpointing")

print("Running:\n", " ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True, cwd=REPO_DIR, env={**os.environ, "HF_TOKEN": os.environ["HF_TOKEN"]})

In [ ]:
import json

print("Output dir:", TRAIN_OUTPUT_DIR)
print("\nTop-level contents:")
for path in sorted(TRAIN_OUTPUT_DIR.glob("*")):
    print(path.name)

final_dir = TRAIN_OUTPUT_DIR / "final"
mapping_path = final_dir / "mimi_token_mapping.json"
train_config_path = final_dir / "train_config.json"

if mapping_path.exists():
    print("\nFinal token mapping:")
    print(json.dumps(json.loads(mapping_path.read_text()), indent=2))

if train_config_path.exists():
    print("\nSaved training config:")
    print(json.dumps(json.loads(train_config_path.read_text()), indent=2))